In [1]:
import os
import numpy as np
import librosa
from tqdm import tqdm
from joblib import Parallel, delayed
import pandas as pd
import glob
BASE_DIR = "/kaggle/input/competitions/birdclef-2026"
AUDIO_DIR=os.path.join(BASE_DIR,"train_audio/") 
df = pd.read_csv('/kaggle/input/competitions/birdclef-2026/train.csv')
file_paths=df['filename'].apply(lambda x: os.path.join(AUDIO_DIR, x)).values
unique_birds = sorted(df['primary_label'].unique())
bird_to_idx = {name: i for i, name in enumerate(unique_birds)}
SAVE_DIR = "/kaggle/working/processed_specs/"
os.makedirs(SAVE_DIR, exist_ok=True)

In [2]:
import numpy as np
import ast

# --- PART A: Define the Mapping ---
unique_birds = sorted(df['primary_label'].unique())
bird_to_idx = {name: i for i, name in enumerate(unique_birds)}
num_classes = len(unique_birds)

# --- PART B: Create the Matrix ---
num_files = len(file_paths)
labels_matrix = np.zeros((num_files, num_classes), dtype=np.float32)

# --- PART C: Run the Loop ---
for i, path in enumerate(file_paths):
    # Match path to CSV
    fname = "/".join(path.split('/')[-2:])
    row = df[df['filename'] == fname].iloc[0]
    
    # Add Primary
    p_bird = row['primary_label']
    labels_matrix[i, bird_to_idx[p_bird]] = 1.0
    
    # Add Secondaries
    s_birds_list = ast.literal_eval(row['secondary_labels'])
    for s_bird in s_birds_list:
        if s_bird in bird_to_idx:
            labels_matrix[i, bird_to_idx[s_bird]] = 1.0

print("Labels Matrix successfully created!")

Labels Matrix successfully created!


In [3]:
import os
import numpy as np
import librosa
from tqdm import tqdm
from joblib import Parallel, delayed

# 1. Setup Directories
SAVE_DIR = "/kaggle/working/processed_specs/"
os.makedirs(SAVE_DIR, exist_ok=True)

# 2. Define the Complete Worker (No external dependencies)
def process_one_file(path, label_row):
    # Match naming logic
    folder_name = path.split('/')[-2]
    file_name = path.split('/')[-1].replace('.ogg', '.npy')
    save_path = os.path.join(SAVE_DIR, f"{folder_name}_{file_name}")
    
    # Skip if already exists (Resume capability)
    if os.path.exists(save_path):
        return save_path, label_row
        
    try:
        # Load audio (Limit to 7s to prevent Disk Full errors)
        audio, _ = librosa.load(path, sr=32000, duration=7.0)
        
        if len(audio) < 2048:
            return None, None
            
        # Generate Spectrogram
        spec = librosa.feature.melspectrogram(y=audio, sr=32000, n_mels=128, fmin=40, fmax=15000)
        spec_db = librosa.power_to_db(spec, ref=np.max)
        
        # Normalize to uint8 (Saves 75% Disk Space)
        spec_db = ((spec_db - spec_db.min()) / (spec_db.max() - spec_db.min()) * 255).astype(np.uint8)
        
        np.save(save_path, spec_db)
        return save_path, label_row
    except Exception:
        return None, None

# 3. Run Parallel Processing
print("🚀 Starting Parallel Preprocessing (using all CPU cores)...")

# We pass BOTH path and labels_matrix row to keep them in sync
results = Parallel(n_jobs=-1)(
    delayed(process_one_file)(p, l) for p, l in tqdm(zip(file_paths, labels_matrix), total=len(file_paths))
)

# 4. Filter and Update Reference
final_npy_paths = []
final_labels_list = []

for npy_path, label in results:
    if npy_path is not None:
        final_npy_paths.append(npy_path)
        final_labels_list.append(label)

# Convert to final arrays for the Model
final_npy_paths = np.array(final_npy_paths)
final_labels_matrix = np.vstack(final_labels_list)

print(f"✅ Finished! Cleaned Dataset size: {len(final_npy_paths)}")
# 1. Save the "Address Book" (Where the spectrograms are)
np.save('final_paths.npy', final_npy_paths)

# 2. Save the "Answers" (The 206-class 1s and 0s)
np.save('final_labels.npy', final_labels_matrix)

# 3. Save the bird mapping (Optional but very helpful for submission)
# This keeps the order of the 206 birds consistent
np.save('bird_names.npy', np.array(unique_birds))

print("✅ SUCCESS: Reference files saved to /kaggle/working/")
print(f"Saved: final_paths.npy, final_labels.npy, bird_names.npy")

🚀 Starting Parallel Preprocessing (using all CPU cores)...


100%|██████████| 35549/35549 [08:03<00:00, 73.55it/s]


✅ Finished! Cleaned Dataset size: 35461
✅ SUCCESS: Reference files saved to /kaggle/working/
Saved: final_paths.npy, final_labels.npy, bird_names.npy


In [4]:

ss_df = pd.read_csv('/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv')
SS_AUDIO_DIR = "/kaggle/input/competitions/birdclef-2026/train_soundscapes/"

def process_soundscape_file(filename, group, bird_to_idx, num_classes):
    path = os.path.join(SS_AUDIO_DIR, filename)
    local_paths = []
    local_labels = []
    
    try:
        audio, _ = librosa.load(path, sr=32000)
        for _, row in group.iterrows():
            # Get the 5s window start (e.g., from '00:00:05' to 5)
            start_sec = int(row['start'].split(':')[-1])
            
            # Extract 7s chunk centered on the 5s mark
            start_frame = int((start_sec - 1.0) * 32000) 
            end_frame = start_frame + int(7.0 * 32000)
            
            # Handle padding for start/end of file
            if start_frame < 0:
                chunk = np.pad(audio[0:end_frame], (abs(start_frame), 0), mode='reflect')
            elif end_frame > len(audio):
                chunk = np.pad(audio[start_frame:], (0, end_frame - len(audio)), mode='reflect')
            else:
                chunk = audio[start_frame:end_frame]
            
            # Generate and normalize spectrogram (matching your uint8 logic)
            spec = librosa.feature.melspectrogram(y=chunk, sr=32000, n_mels=128, fmin=40, fmax=15000)
            spec_db = librosa.power_to_db(spec, ref=np.max)
            spec_db = ((spec_db - spec_db.min()) / (spec_db.max() - spec_db.min() + 1e-6) * 255).astype(np.uint8)
            
            save_path = os.path.join(SAVE_DIR, f"ss_{filename}_{start_sec}.npy")
            np.save(save_path, spec_db)
            
            # Create Multi-Hot Label
            label_vec = np.zeros(num_classes, dtype=np.float32)
            birds = str(row['primary_label']).split(';')
            for b in birds:
                if b in bird_to_idx:
                    label_vec[bird_to_idx[b]] = 1.0
            
            local_paths.append(save_path)
            local_labels.append(label_vec)
            
        return local_paths, local_labels
    except Exception as e:
        return [], []

# Execute Soundscape Processing
print("🚀 Slicing 60s Soundscapes into 7s segments...")
ss_results = Parallel(n_jobs=-1)(
    delayed(process_soundscape_file)(fn, grp, bird_to_idx, num_classes) 
    for fn, grp in tqdm(ss_df.groupby('filename'))
)

# Merge with your existing final_npy_paths and final_labels_matrix
for p_list, l_list in ss_results:
    if p_list:
        final_npy_paths = np.append(final_npy_paths, p_list)
        final_labels_matrix = np.vstack([final_labels_matrix, l_list])

# Re-save the final reference files
np.save('final_paths.npy', final_npy_paths)
np.save('final_labels.npy', final_labels_matrix)
print(f"✅ Combined Dataset Size: {len(final_npy_paths)}")

🚀 Slicing 60s Soundscapes into 7s segments...


100%|██████████| 66/66 [00:15<00:00,  4.25it/s]


✅ Combined Dataset Size: 36939
